<a href="https://colab.research.google.com/github/pika97/Predictive-Maintenence---Machine-Failure-Prediction/blob/main/MLmodel_for_machiner_failure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The Bussiness Problem: The manufacturing company that operates hundreds of machines.

Machines sometimes fail unexpectedly, causing:
- Production delays.
- Increased maintenance costs.
- Equipment downtime.
- Product quality issues.
- Potential safety risks.

Predictive Maintenance: Machine Failure Prediction Using Machine Learning

Project Overview
This project focuses on developing a supervised machine learning model to predict machine failures using industrial operating data. The objective is to identify potential equipment failures based on parameters such as air temperature, process temperature, rotational speed, torque and tool wear.

I chose predictive maintenance because unexpected equipment failures can lead to production downtime, increased maintenance costs and operational inefficiencies. With my background in Biomedical Engineering and experience in quality investigations, I wanted to explore how machine learning can be applied to equipment reliability, failure prediction and preventive decision-making.

I selected supervised learning because the AI4I 2020 dataset contains labelled historical observations, including a `Machine Failure` target indicating whether a failure occurred (0 = No Failure, 1 = Failure). This allows a classification model to learn patterns from labelled data and predict failures in previously unseen observations.

Python Libraries and Methods Used
- Python: Main programming language.
- Pandas & NumPy: Data loading, exploration, cleaning and numerical operations.
- Matplotlib: Exploratory data analysis and data visualisation.
- Scikit-learn: Data preprocessing, feature engineering, model training and evaluation.
- Data Cleaning: Missing-value handling, duplicate checks and outlier investigation.
- Preprocessing: Median imputation, feature scaling using StandardScaler and categorical encoding using OneHotEncoder.
- Data Splitting: 80% training and 20% testing.
- Machine Learning Models: Logistic Regression and Random Forest Classifier.
- Model Evaluation: Accuracy, Precision, Recall, F1-score, ROC-AUC, PR-AUC and Confusion Matrix.

Project Objective
To build, evaluate and compare supervised classification models, identify important machine operating parameters associated with failures, and explore how predictive analytics can support proactive equipment maintenance.

Task as an Ai/ML Engineer is to build a machine learning model that predicts weather a machine is likely to experience a failure.

In [2]:
# data collection and loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# splitting and validating
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, cross_val_predict

# pre-processing tools
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# evaluation metrics
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
confusion_matrix, PrecisionRecallDisplay, precision_recall_curve)

# feature importance
from sklearn.inspection import permutation_importance

# warning handling
from sklearn.exceptions import UndefinedMetricWarning
from pathlib import path

warnings.filterwarnings("ignore", UndefinedMetricWarnings)

RANDOM_STATE = 42

# Load and read the dataset
d_f = pd.read_csv('/content/ai4i+2020+predictive+maintenance+dataset (1).zip')
df.columns = df.columns.str.strip()
df.shape()
df.head()


# Data exploration and analysis (EDA)
Product_id = (d_f['Product ID'].duplicated().sum())
print("\nDuplicate_Product_id's:\n", Product_id)
print("\nMissing values:\n", d_f.isnull().sum())
print("\nDuplicated values:\n", d_f.duplicated().sum())
display(d_f.dtypes)

failure_rate = d_f["Machine failure"].mean()
print(f"Failure rate: {failure_rate:.2%}")
print(f"A model that always predicts no failure would be {1 - failure_rate:.2%} accurate.")

process_parameters_col = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, col in zip(axes, process_parameters_col):
    ax.hist(df.loc[df["Machine Failure"] == 0, col], bins=30, alpha=0.5, density=True, label="No Failure")
    ax.hist(df.loc[df["Machine Failure"] == 1, col], bins=30, alpha=0.5, density=True, label="Failure")
    ax.set_title(col, fontsize=9)
axes[0].legend
plt.tight_layout()
plt.show()

# separate X and y features
X = d_f[["Type", "Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]]
y = d_f["Machine failure"]

# One-hot encode the categorical 'Type' feature
X = pd.get_dummies(X, columns=["Type"], drop_first=True)     # turns category ('type'(H, M, L) in 0/1 columns)

# splitting the data in training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\nTrain shape:\n", X_train.shape, y_train.shape)
print("\nTest shape:\n", X_test.shape, y_test.shape, f"| failures in test: {y_test.sum()}")







# look at the ranges in the data
display(X_train.describe().loc[['min', 'max', 'mean', 'std']])

# Identify numerical features in X
features_numeric = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

# scaling the data
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[features_numeric] = scaler.fit_transform(X_train[features_numeric])
X_test_scaled[features_numeric] = scaler.transform(X_test[features_numeric])

display("X_train_scaled:", X_train_scaled.head())
display("X_test_scaled:", X_test_scaled.head())

# looking for outliers in numerical columns (excluding non-numeric columns)
numeric_cols = d_f.select_dtypes(include=[np.number])
Q1 = np.percentile(numeric_cols, 25, axis=0)
Q3 = np.percentile(numeric_cols, 75, axis=0)
IQR = Q3 - Q1
Lower_bound = Q1 - 1.5 * IQR
Upper_bound = Q3 + 1.5 * IQR

# Compare against numeric_cols instead of d_f to match dimensions, and use d_f consistently
Outlier = numeric_cols[(numeric_cols < Lower_bound) | (numeric_cols > Upper_bound)]
display("\nOutliers:\n", Outlier.dropna(how='all'))
print("\nOutlier Count:\n", Outlier.count())

# BUILDING THE MACHINE LEARNING MODEL USING RANDOM FOREST CLASSIFIER

# create a Random Forest Model
odel = RandomForestClassifier(n_estimators=1000, random_state=42)

# train model
model.fit(X_train_scaled, y_train)

# make prediction using scaled test features
prediction = model.predict(X_temst_scaled)
print("\nPrediction:\n", prediction)

# check accuracy using scaled test features
accuracy = model.score(X_test_scaled, y_test)
display("Accuracy:", accuracy)

ImportError: cannot import name 'path' from 'pathlib' (/usr/lib/python3.13/pathlib/__init__.py)